# Lección 13.2 — Autoencoders y VAEs: comprimir y reconstruir

> **Módulo 13 · Modelos generativos y LLMs** · Nivel 5 — Héroe
> ⏱️ Tiempo estimado: 100–120 min (lección con ejercicios intercalados ≈ 75 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 8.4 (PCA), Lección 9.6 (hiperparámetros de la red), Lección 13.1 (discriminativo vs. generativo)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar con una analogía qué hace un **autoencoder**: comprimir (codificar) y reconstruir (decodificar).
- Relacionar el autoencoder con **PCA** y ver por qué una red no lineal comprime mejor.
- Entender qué es el **espacio latente** y por qué un autoencoder normal **no** sirve para generar.
- Explicar cómo un **VAE** ordena el espacio latente para poder **crear** imágenes nuevas, y el papel de β.
- Usar el error de reconstrucción para **detectar anomalías** y para **quitar ruido**.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas y un subconjunto de MNIST (dígitos escritos a mano de 28×28 píxeles). La primera vez puede tardar un poco en descargarse.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from sklearn.decomposition import PCA

device = "cuda" if torch.cuda.is_available() else "cpu"
plt.rcParams["figure.figsize"] = (8, 4)

entrenamiento = datasets.MNIST(root="data", train=True, download=True, transform=transforms.ToTensor())
prueba = datasets.MNIST(root="data", train=False, download=True, transform=transforms.ToTensor())

# Para que todo corra rápido en CPU usamos 10,000 imágenes de entrenamiento y 2,000 de prueba.
# Cada imagen se "aplana" en una fila de 784 números entre 0 (negro) y 1 (blanco).
X = entrenamiento.data[:10_000].float().div(255).reshape(-1, 784)
y = entrenamiento.targets[:10_000]
X_prueba = prueba.data[:2_000].float().div(255).reshape(-1, 784)

def mostrar(imagenes, titulo="", n=10):
    """Dibuja n imágenes en una fila."""
    imagenes = torch.as_tensor(np.asarray(imagenes, dtype="float32"))[:n].reshape(-1, 28, 28)
    fig, axes = plt.subplots(1, len(imagenes), figsize=(1.2 * len(imagenes), 1.5))
    for ax, img in zip(np.atleast_1d(axes), imagenes):
        ax.imshow(img, cmap="gray", vmin=0, vmax=1); ax.axis("off")
    fig.suptitle(titulo, fontsize=10); plt.show()

print(f"✅ Todo listo · {len(X)} imágenes de entrenamiento con {X.shape[1]} píxeles cada una")
mostrar(X, "Algunos dígitos originales")

---
## 1. 📖 La intuición: dictar un dibujo por teléfono

Imagina que tienes que describirle a un amigo, **por teléfono**, un dígito escrito a mano para que él lo dibuje. No puedes dictarle los 784 píxeles uno por uno: solo te dejan decir **unos pocos números**.

```
  🖼️ dígito  ──►  🗣️ tú resumes  ──►  "un 7, inclinado, trazo grueso"  ──►  👂 tu amigo  ──►  🖼️ dibujo
  (784 píxeles)    (CODIFICADOR)        (pocos números = CÓDIGO)              (DECODIFICADOR)   (reconstrucción)
```

Si los dos se ponen de acuerdo en **qué números importan** ("qué dígito es", "cuánto se inclina", "qué tan grueso"), el dibujo final se parecerá mucho al original.

**Un autoencoder es exactamente eso:** dos redes neuronales que aprenden juntas a **comprimir** una imagen en pocos números y a **reconstruirla** a partir de ellos.

### ✋ Ejercicio 1 — Calentamiento
Si solo pudieras dictar **2 números** por dígito, ¿qué dos cosas elegirías describir? ¿Crees que tu amigo podría reconstruir bien cualquier dígito con solo esos 2 números?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

No hay una única respuesta. Algo razonable: "qué dígito es" y "cuánto se inclina", o "qué tan redondo" y "qué tan grueso". Con solo 2 números el dibujo será **aproximado**: se perderán detalles como el ganchito del 7 o una mancha de tinta.

Esa es la tensión central de esta lección: **menos números = más compresión, pero peor reconstrucción.**
</details>

---
## 2. 📖 El concepto: una red que se copia a sí misma

```
   entrada x                                                   reconstrucción x̂
  (784 píxeles) ─► [ CODIFICADOR ] ─► ● ● (código z) ─► [ DECODIFICADOR ] ─► (784 píxeles)
                    784 → 256 → 2     "cuello de botella"   2 → 256 → 784
```

| Término | Qué es | En la analogía |
|---|---|---|
| **Codificador** (*encoder*) | Red que convierte la imagen en pocos números | Tú resumiendo por teléfono |
| **Código / vector latente** (*latent vector*, *z*) | Esos pocos números | "un 7, inclinado, grueso" |
| **Espacio latente** (*latent space*) | El "mapa" donde vive cada código | Todas las descripciones posibles |
| **Decodificador** (*decoder*) | Red que reconstruye la imagen desde el código | Tu amigo dibujando |
| **Error de reconstrucción** | Qué tan distinta quedó la copia del original | Qué tan mal le salió el dibujo |

**La trampa genial:** la "respuesta correcta" que debe producir la red es **la propia imagen de entrada**. No hacen falta etiquetas: los datos se supervisan a sí mismos (*aprendizaje autosupervisado*). Como el cuello de botella es estrecho, la red **no puede** copiar los píxeles tal cual: está obligada a descubrir lo esencial.

### ✋ Ejercicio 2 — ¿Qué tipo de aprendizaje es?
Para entrenar el autoencoder usamos `pérdida = error(reconstrucción, imagen_original)`.
1. ¿Necesitamos saber qué dígito es cada imagen (la etiqueta `y`)?
2. ¿Qué pasaría si el cuello de botella tuviera **784** números, igual que la entrada?

✍️ *Tus respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **No.** El objetivo es la propia imagen. Por eso se usa mucho cuando hay montañas de datos sin etiquetar (Lección 8.1).
2. La red podría aprender a **copiar** sin entender nada (la "función identidad"): error casi cero, pero un código inútil. El cuello de botella estrecho es lo que obliga a **resumir**.
</details>

---
## 3. 📖 Cómo funciona por dentro

### 3.1 Primero "a mano": PCA ya es un autoencoder (lineal)
En la Lección 8.4 viste que **PCA** busca las pocas direcciones que mejor resumen los datos. Fíjate que hace justo lo de un autoencoder:
- `transform` = **codificar** (784 píxeles → *k* números),
- `inverse_transform` = **decodificar** (*k* números → 784 píxeles).

La diferencia: PCA solo puede usar **combinaciones lineales** (sumas ponderadas). Veamos qué tan bien reconstruye.

### 🔮 Ejercicio 3 — Predice antes de ejecutar
Vamos a comprimir con PCA usando **2**, **10** y **32** números. Ordena del mejor al peor según el error de reconstrucción en prueba. ¿Con cuántos números crees que el dígito ya se reconoce bien?

✍️ *Mi predicción:*

In [ ]:
mostrar(X_prueba, "Originales (prueba)")
for k in [2, 10, 32]:                      # ✏️ prueba otros valores, p. ej. 1 o 100
    pca = PCA(n_components=k, random_state=0).fit(X.numpy())
    codigo = pca.transform(X_prueba.numpy())          # codificar: 784 → k números
    reconstruccion = pca.inverse_transform(codigo)    # decodificar: k → 784
    error = ((reconstruccion - X_prueba.numpy()) ** 2).mean()
    mostrar(reconstruccion.clip(0, 1), f"PCA con {k} números · error = {error:.4f}")

<details><summary><b>👀 Ver respuesta</b></summary>

32 (error ≈ 0.018) es mejor que 10 (≈ 0.034), que es mejor que 2 (≈ 0.054). Con 2 números casi todo sale como una mancha borrosa parecida a un 9, un 0 o un 1; con 10 ya se adivinan varios (el 7, el 0, el 1, el 4) y con 32 se reconoce la mayoría.

Fíjate: comprimir 784 → 32 es guardar **menos del 5 %** de los números y aun así reconocer el dígito. Las imágenes tienen mucha información **redundante**.
</details>

### 3.2 Ahora con una red neuronal (PyTorch)
Un autoencoder cambia las sumas lineales de PCA por **redes con activaciones no lineales** (Lección 9.3): puede doblar y curvar el espacio para resumir mejor.

La función de abajo construye y entrena un autoencoder pequeño. No necesitas entender cada línea: fíjate en los comentarios. Más adelante la reutilizaremos para el VAE (por eso tiene opciones `vae` y `beta`) y para quitar ruido (opción `ruido`).

In [ ]:
def entrenar_autoencoder(dim_latente=2, epocas=15, vae=False, beta=1.0, lr=1e-3, ruido=0.0, semilla=0):
    torch.manual_seed(semilla)
    # Codificador: 784 → 256 → código.  (El VAE produce el doble: una media y una "dispersión" por número.)
    codificador = nn.Sequential(nn.Linear(784, 256), nn.ReLU(),
                                nn.Linear(256, 2 * dim_latente if vae else dim_latente)).to(device)
    # Decodificador: código → 256 → 784, con sigmoide para que los píxeles queden entre 0 y 1
    decodificador = nn.Sequential(nn.Linear(dim_latente, 256), nn.ReLU(),
                                  nn.Linear(256, 784), nn.Sigmoid()).to(device)
    optimizador = torch.optim.Adam([*codificador.parameters(), *decodificador.parameters()], lr=lr)
    for epoca in range(epocas):
        orden = torch.randperm(len(X))
        for i in range(0, len(X), 128):                      # lotes de 128 imágenes
            limpio = X[orden[i:i + 128]].to(device)
            entrada = (limpio + ruido * torch.randn_like(limpio)).clamp(0, 1)   # opción para quitar ruido
            h = codificador(entrada)
            if vae:
                media, log_var = h.chunk(2, dim=1)
                z = media + torch.randn_like(media) * (0.5 * log_var).exp()   # muestreo con "temblor"
                reconstruccion = F.binary_cross_entropy(decodificador(z), limpio, reduction="sum") / len(limpio)
                kl = -0.5 * (1 + log_var - media**2 - log_var.exp()).sum() / len(limpio)  # "ordena el mapa"
                perdida = reconstruccion + beta * kl
            else:
                perdida = ((decodificador(h) - limpio) ** 2).mean()   # error cuadrático medio
            optimizador.zero_grad(); perdida.backward(); optimizador.step()
    return codificador.eval(), decodificador.eval()

@torch.no_grad()
def reconstruir(codificador, decodificador, imagenes, vae=False):
    h = codificador(imagenes.to(device))
    z = h.chunk(2, dim=1)[0] if vae else h           # en el VAE usamos la media (sin temblor)
    return decodificador(z).cpu()

def error_reconstruccion(codificador, decodificador, imagenes, vae=False):
    return ((reconstruir(codificador, decodificador, imagenes, vae) - imagenes) ** 2).mean().item()

print("✅ Funciones listas")

### 🔮 Ejercicio 4 — ¿Red o PCA?
Entrenamos un autoencoder con un código de solo **2 números** (15 épocas, unos segundos). Con PCA de 2 números el error fue ≈ 0.054.
- ¿El autoencoder tendrá un error **mayor, igual o menor**?
- ¿Se verán más nítidos los dígitos?

✍️ *Mi predicción:*

In [ ]:
cod_ae, dec_ae = entrenar_autoencoder(dim_latente=2)
error_ae = error_reconstruccion(cod_ae, dec_ae, X_prueba)
mostrar(X_prueba, "Originales")
mostrar(reconstruir(cod_ae, dec_ae, X_prueba), f"Autoencoder con 2 números · error = {error_ae:.4f}")

<details><summary><b>👀 Ver respuesta</b></summary>

El autoencoder gana: error ≈ **0.046** frente a ≈ 0.054 de PCA, con el mismo presupuesto de 2 números. Las reconstrucciones se ven algo más nítidas que las de PCA, pero siguen siendo borrosas y los dos se equivocan de dígito a menudo (el 7 sale como un 9, el 2 como un 3): con solo 2 números no cabe todo.

La razón: la red es **no lineal**, así que puede aprovechar mejor esos 2 números. Con la misma idea, pero con códigos más grandes, la diferencia crece (lo verás en la Sección 5).
</details>

### 📖 El espacio latente: un mapa de dígitos
Como el código tiene solo 2 números, podemos **dibujar** cada imagen como un punto en un plano. El autoencoder **nunca vio las etiquetas**; las usamos solo para colorear.

In [ ]:
with torch.no_grad():
    z_ae = cod_ae(X[:3000].to(device)).cpu()
plt.figure(figsize=(7, 6))
puntos = plt.scatter(z_ae[:, 0], z_ae[:, 1], c=y[:3000], cmap="tab10", s=6)
plt.colorbar(puntos, label="dígito (solo para colorear)")
plt.title("Espacio latente del autoencoder (2 números por imagen)"); plt.xlabel("z₁"); plt.ylabel("z₂"); plt.show()

### ✋ Ejercicio 5 — Lee el mapa
1. ¿Los dígitos iguales tienden a quedar juntos, aunque la red nunca supo qué dígito era cada uno?
2. ¿Qué dígitos se mezclan más entre sí? ¿Tiene sentido por cómo se escriben?
3. Fíjate en los ejes: ¿la nube está centrada en (0, 0)? ¿Tiene "huecos"?

✍️ *Tus respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Sí.** Se forman regiones por dígito: para reconstruir bien, a la red le conviene poner cerca las imágenes parecidas. Aprendió algo de "qué dígito es" sin que se lo dijéramos.
2. Los que se escriben parecido se enciman: **4, 9 y 7** comparten una gran zona, y **2, 3, 5 y 8** se mezclan en el centro. El **0** y el **1** quedan en zonas propias (un círculo y un palito son muy distintos).
3. La nube **no** está centrada ni tiene una forma ordenada: se estira en "brazos" de tamaños muy distintos y deja huecos. Guarda esta observación: es el problema de la siguiente sección.
</details>

---
## 4. 📖 ¿Qué pasa si… intento generar dígitos nuevos?

Esta es la gran pregunta de este módulo (Lección 13.1): ¿podemos usar el decodificador como **máquina de crear**? La idea sería: elegir un punto **al azar** en el espacio latente y decodificarlo.

### 🔮 Ejercicio 6 — Predice
Tomamos 10 puntos al azar de una nube "estándar" (centrada en 0, ancho ≈ 1) y se los damos al **decodificador del autoencoder**. ¿Qué esperas ver: dígitos nítidos y variados, dígitos borrosos o garabatos?

✍️ *Mi predicción:*

In [ ]:
torch.manual_seed(1)
z_azar = torch.randn(10, 2)          # 10 puntos al azar alrededor de (0, 0)
with torch.no_grad():
    mostrar(dec_ae(z_azar.to(device)).cpu(), "Autoencoder: decodificando puntos al azar")
print("Rango real de los códigos del autoencoder:")
print("  z₁ de", round(z_ae[:, 0].min().item(), 1), "a", round(z_ae[:, 0].max().item(), 1))
print("  z₂ de", round(z_ae[:, 1].min().item(), 1), "a", round(z_ae[:, 1].max().item(), 1))

<details><summary><b>👀 Ver respuesta</b></summary>

Salen **manchas borrosas y muy parecidas entre sí** (casi todas una especie de 6 o un garabato), sin la variedad de los dígitos reales. El problema está en los rangos impresos: los códigos reales del autoencoder ocupan una zona **mucho más amplia y deformada** (z₁ de ≈ −6 a ≈ 18, z₂ de ≈ −13 a ≈ 7) que la nube estándar alrededor de (0, 0). Los puntos al azar caen todos en un rincón del mapa, justo donde se mezclan varios dígitos, o en **huecos** que el decodificador nunca vio.

El autoencoder aprendió a **comprimir**, no a **crear**: su mapa no tiene un orden pensado para que cualquier punto signifique algo.
</details>

### 📖 La solución: el autoencoder variacional (VAE)
Un **VAE** (*Variational Autoencoder*) cambia dos cosas:

1. **Código con temblor.** El codificador ya no da un punto exacto, sino una **zona**: una media y una dispersión. Durante el entrenamiento se toma un punto al azar dentro de esa zona. Así el decodificador aprende que **los vecinos de un código también deben dar dígitos válidos**: se rellenan los huecos.
2. **Un castigo para ordenar el mapa (KL).** Se añade a la pérdida un término (la *divergencia KL*) que empuja todas las zonas hacia una nube estándar centrada en (0, 0). Es como un bibliotecario que obliga a guardar todos los libros en **una sola estantería compacta**, sin huecos.

```
pérdida del VAE = error de reconstrucción  +  β × KL ("qué tan desordenado está el mapa")
```

El número **β** (beta) decide cuánto pesa el orden frente a la fidelidad. Con β = 1 es el VAE clásico.

> 💡 Analogía: el autoencoder guarda cada libro en un lugar exacto pero caótico; el VAE guarda cada libro "más o menos en esta zona" y todas las zonas en la misma estantería. Si eliges un estante al azar, **casi siempre hay un libro**.

### 🔮 Ejercicio 7 — El VAE al rescate
Entrenamos un VAE con 2 números (20 épocas, unos segundos) y repetimos el experimento: 10 puntos al azar de la nube estándar.
1. ¿Mejorarán las imágenes generadas?
2. ¿El error de reconstrucción del VAE será mejor o peor que el del autoencoder (≈ 0.046)?

✍️ *Mis predicciones:*

In [ ]:
cod_vae, dec_vae = entrenar_autoencoder(dim_latente=2, epocas=20, vae=True, beta=1.0)
print(f"Error de reconstrucción del VAE: {error_reconstruccion(cod_vae, dec_vae, X_prueba, vae=True):.4f}")
with torch.no_grad():
    mostrar(dec_vae(z_azar.to(device)).cpu(), "VAE: decodificando LOS MISMOS puntos al azar")

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Sí:** ahora casi todos los puntos al azar dan **dígitos reconocibles** (algo borrosos, porque son solo 2 números). Hemos creado dígitos que **no existen** en los datos: esto ya es un modelo **generativo**.
2. El error de reconstrucción es **casi igual o apenas peor** (≈ 0.046–0.047). Es el precio del orden: el VAE sacrifica un poquito de fidelidad a cambio de un mapa en el que se puede **muestrear**.
</details>

### ▶️ Paseo por el espacio latente
Como el mapa del VAE está ordenado, podemos recorrerlo en una **rejilla** y decodificar cada punto. Mira cómo un dígito se **transforma suavemente** en otro.

In [ ]:
valores = torch.linspace(-2, 2, 12)
fig, axes = plt.subplots(12, 12, figsize=(7, 7))
with torch.no_grad():
    for i, z2 in enumerate(valores.flip(0)):
        for j, z1 in enumerate(valores):
            img = dec_vae(torch.tensor([[z1, z2]], device=device)).cpu().reshape(28, 28)
            axes[i, j].imshow(img, cmap="gray"); axes[i, j].axis("off")
plt.suptitle("Rejilla del espacio latente del VAE (z₁ → horizontal, z₂ → vertical)"); plt.show()

### ✋ Ejercicio 8 — Interpolar
En la rejilla, elige dos dígitos distintos que estén en la misma fila. ¿Qué ves entre ellos: un salto brusco o una transición gradual? ¿Para qué podría servir esto en un producto real (piensa en caras, fuentes tipográficas o diseños)?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Una **transición gradual**: los dígitos intermedios mezclan rasgos de ambos (por ejemplo, un 0 que se va cerrando hasta ser un 8 y luego se estira hasta ser un 1, o un 9 que se convierte poco a poco en un 7). Como cada zona del mapa está ocupada, moverse por él produce cambios suaves.

Usos: generar **variaciones** de un diseño, mezclar dos estilos de letra, "envejecer" una cara moviéndose en una dirección del latente, o rellenar datos faltantes. Es la misma idea que verás amplificada en los modelos de difusión (Lección 13.4).
</details>

---
## 5. 📖 Hiperparámetros y perillas

| Hiperparámetro | Qué controla | Si lo **subes** ⬆️ | Si lo **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| `dim_latente` | Cuántos números tiene el código | Mejor reconstrucción, menos compresión; en un VAE, más difícil de visualizar | Más compresión, reconstrucción más borrosa | 2 (para ver) · 16–128 (para usar) |
| `beta` (β, solo VAE) | Peso del orden del mapa (KL) frente a la fidelidad | Mapa más ordenado y bueno para generar, pero imágenes más borrosas | Reconstrucciones más fieles, pero el mapa vuelve a tener huecos (se parece a un autoencoder) | 1 |
| Tamaño de las capas | Capacidad del codificador/decodificador | Aprende formas más complejas; riesgo de sobreajuste y más lento | Más rápido, menos detalle | 256–512 neuronas |
| `epocas` | Cuántas pasadas por los datos | Mejor ajuste hasta cierto punto | Reconstrucciones a medio aprender | 10–50 |
| `lr` (tasa de aprendizaje) | Tamaño de cada paso de Adam (Lección 9.5) | Más rápido pero inestable | Más lento | 1e-3 |
| `ruido` (autoencoder que quita ruido) | Cuánto se ensucia la entrada al entrenar | Aprende a limpiar ruido fuerte; puede perder detalle | Se parece a un autoencoder normal | 0.2–0.5 |

### 🔮 Ejercicio 9 — La perilla `dim_latente`
Vamos a entrenar autoencoders (no VAE) con códigos de **2, 8 y 32** números. Antes de ejecutar:
1. ¿Cómo cambiará el error?
2. Con 32 números, ¿le ganará el autoencoder a PCA con 32 (≈ 0.018)?

✍️ *Mis predicciones:* (tarda ≈ 20 s)

In [ ]:
for d in [2, 8, 32]:                 # ✏️ prueba también 64
    c, dd = entrenar_autoencoder(dim_latente=d)
    err = error_reconstruccion(c, dd, X_prueba)
    mostrar(reconstruir(c, dd, X_prueba), f"Autoencoder con {d} números · error = {err:.4f}")

<details><summary><b>👀 Ver respuesta</b></summary>

1. El error baja al subir la dimensión: ≈ **0.046** (2) → ≈ **0.024** (8) → ≈ **0.012** (32). Con 32 los dígitos ya son casi idénticos a los originales.
2. **Sí:** ≈ 0.012 contra ≈ 0.018 de PCA con el mismo número de componentes. La red no lineal saca más provecho de cada número.

Pero ojo: un código grande comprime menos. En compresión siempre eliges un punto en esta balanza.
</details>

### 🔮 Ejercicio 10 — La perilla β del VAE
Entrenamos tres VAEs con β = **0.1**, **1** y **5** y medimos dos cosas:
- el **error de reconstrucción**,
- qué tan **centrado y compacto** quedó el mapa (media y dispersión de los códigos; el ideal para muestrear es media ≈ 0 y dispersión ≈ 1).

¿Qué β dará el mapa más ordenado? ¿Y la mejor reconstrucción?

✍️ *Mi predicción:* (tarda ≈ 20 s)

In [ ]:
for beta in [0.1, 1, 5]:             # ✏️ prueba 0 (¿a qué se parece?) o 20
    c, dd = entrenar_autoencoder(dim_latente=2, epocas=20, vae=True, beta=beta)
    with torch.no_grad():
        medias = c(X.to(device)).chunk(2, dim=1)[0].cpu()
    err = error_reconstruccion(c, dd, X_prueba, vae=True)
    print(f"β = {beta:<4} error = {err:.4f} · centro del mapa = {medias.mean(0).numpy().round(2)} "
          f"· dispersión = {medias.std(0).numpy().round(2)}")
    with torch.no_grad():
        mostrar(dd(z_azar.to(device)).cpu(), f"β = {beta}: dígitos generados al azar")

<details><summary><b>👀 Ver respuesta</b></summary>

- **β = 0.1:** el mapa se descentra y se estira (centro ≈ (1.3, 1.2), dispersión ≈ 2.2–2.6): se parece más al autoencoder normal y, al muestrear de la nube estándar, aparecen algunos dígitos deformes.
- **β = 1:** el equilibrio clásico (centro cerca de 0, dispersión ≈ 1.3–1.4), con la mejor reconstrucción de los tres (≈ 0.046), casi empatado con β = 0.1.
- **β = 5:** el mapa más ordenado (centro ≈ 0, dispersión ≈ 1.0), pero la reconstrucción empeora (≈ 0.048) y los dígitos se ven más "promedio" y borrosos.

(Con un código de solo 2 números las diferencias de error son pequeñas; con códigos más grandes el efecto de β sobre la nitidez es mucho más visible.)

La regla: **β alto = orden (bueno para generar); β bajo = fidelidad (bueno para reconstruir).**
</details>

---
## 6. 📖 Errores comunes y un uso muy práctico: detectar anomalías

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| La reconstrucción es perfecta pero el código no sirve para nada | Cuello de botella demasiado ancho: la red copia | Bajar `dim_latente` o añadir ruido a la entrada |
| Todas las reconstrucciones son el "dígito promedio" borroso | Red muy pequeña, pocas épocas o β demasiado alto | Más épocas, capas más anchas, bajar β |
| Al muestrear al azar salen garabatos | Usas un autoencoder normal, o un VAE con β muy bajo | Usar VAE con β ≈ 1 |
| Las imágenes generadas son siempre borrosas | Limitación conocida de los VAEs (promedian posibilidades) | Para nitidez: GANs (13.3) o difusión (13.4) |
| Píxeles fuera de [0, 1] o pérdida rara | Falta la sigmoide final o los datos no están normalizados | Normalizar a [0, 1] y usar `Sigmoid` al final |

### 📖 Detección de anomalías por error de reconstrucción
Un autoencoder aprende a reconstruir bien **lo que se parece a sus datos de entrenamiento**. Si le das algo muy distinto, lo reconstruirá **mal**. Por eso el error de reconstrucción funciona como **alarma de "esto es raro"** (compáralo con Isolation Forest, Lección 8.6).

### 🔮 Ejercicio 11 — ¿Quién hace sonar la alarma?
Le damos al autoencoder de 32 números tres tipos de imágenes:
- A: dígitos **normales** de prueba,
- B: dígitos en **negativo** (fondo blanco, trazo negro),
- C: **ruido** puro (píxeles al azar).

Ordena de menor a mayor error. ¿Cuántas veces mayor que el normal crees que será el error de las raras?

✍️ *Mi predicción:*

In [ ]:
cod32, dec32 = entrenar_autoencoder(dim_latente=32)
torch.manual_seed(0)
grupos = {
    "A · normales": X_prueba[:500],
    "B · negativos": 1 - X_prueba[:500],
    "C · ruido": torch.rand(500, 784),
}
for nombre, imgs in grupos.items():
    print(f"{nombre:<14} error medio = {error_reconstruccion(cod32, dec32, imgs):.4f}")
mostrar(torch.cat([X_prueba[:3], 1 - X_prueba[:3], torch.rand(3, 784)]), "Entradas: normales · negativos · ruido", n=9)
mostrar(reconstruir(cod32, dec32, torch.cat([X_prueba[:3], 1 - X_prueba[:3], torch.rand(3, 784)])),
        "Lo que reconstruye el autoencoder", n=9)

<details><summary><b>👀 Ver respuesta</b></summary>

A (normales) ≈ **0.012** · C (ruido) ≈ **0.28** · B (negativos) ≈ **0.71**. Las raras tienen un error **entre 20 y 60 veces mayor**: la alarma funciona. (¿Te sorprendió que el negativo sea "más raro" que el ruido? Tiene casi todos los píxeles encendidos, justo lo contrario de un dígito normal.)

Fíjate en las reconstrucciones: los dígitos normales salen casi perfectos, pero ante un negativo o ruido el autoencoder dibuja **manchas sin sentido**: no sabe representar algo que nunca vio. Por eso la diferencia entre entrada y salida es enorme.

En la vida real: se entrena con transacciones, piezas o señales **normales**, y se marca para revisión todo lo que tenga un error de reconstrucción por encima de un umbral (por ejemplo, el percentil 99 del error en datos normales).
</details>

---
## 7. 📋 Ficha del modelo

| | Autoencoder | VAE (autoencoder variacional) |
|---|---|---|
| **Tipo** | No supervisado / autosupervisado | Generativo |
| **Intuición** | Comprimir en pocos números y reconstruir | Igual, pero con un mapa latente ordenado para poder crear |
| **Cómo aprende** | Minimiza el error entre entrada y reconstrucción | Error de reconstrucción + β × KL (orden del mapa) |
| **Hiperparámetros clave** | `dim_latente`, tamaño de capas, `ruido` | `dim_latente`, `beta` |
| **✅ Fortalezas** | Reducción de dimensión no lineal · Detección de anomalías · Quitar ruido | Genera datos nuevos · Interpolación suave · Entrenamiento estable |
| **❌ Debilidades** | No sirve para generar · El código puede ser poco interpretable | Imágenes borrosas · Menos nítido que GANs o difusión |
| **Úsalo cuando…** | Quieres comprimir, limpiar o detectar lo raro | Quieres generar variaciones o un espacio latente "navegable" |
| **Evítalo cuando…** | Necesitas generar | Necesitas imágenes fotorrealistas (usa difusión, 13.4) |

> 💡 Dato real: Stable Diffusion usa **por dentro** un autoencoder (un VAE) para comprimir las imágenes antes de aplicar la difusión. Las ideas de esta lección siguen vivas en los modelos más modernos.

## 8. 🧠 Resumen en 5 puntos
1. Un **autoencoder** = codificador (comprime) + decodificador (reconstruye), entrenado para **copiarse a sí mismo** a través de un **cuello de botella**.
2. **PCA** es un autoencoder **lineal**; la red no lineal comprime mejor con los mismos números.
3. El **espacio latente** agrupa ejemplos parecidos sin usar etiquetas, pero en un autoencoder normal está desordenado y con huecos: **no sirve para generar**.
4. Un **VAE** añade temblor al código y un castigo **KL** que ordena el mapa; así, decodificar un punto al azar produce un ejemplo **nuevo y válido**. **β** equilibra orden y fidelidad.
5. El **error de reconstrucción** es una herramienta práctica: alto = **anomalía**; entrenar con entradas ruidosas enseña a **quitar ruido**.

---
## 9. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas** (≈30 min).

### 9.1 Repaso rápido (opción múltiple)

**1.** ¿Cuál es la "respuesta correcta" que se le da a un autoencoder al entrenarlo?
- a) La etiqueta del dígito · b) La propia imagen de entrada · c) Una imagen al azar · d) Ninguna

**2.** Tu autoencoder reconstruye perfecto, pero el código de 700 números no te sirve para nada. ¿Qué pasa?
- a) Subajuste · b) El cuello de botella es demasiado ancho: la red aprendió a copiar · c) Falta β · d) La tasa de aprendizaje es baja

**3.** Decodificas puntos al azar con un autoencoder **normal** y salen garabatos. ¿Por qué?
- a) Le faltan épocas · b) Su espacio latente tiene huecos y no está centrado · c) Las imágenes son muy pequeñas · d) Por la sigmoide

**4.** En un VAE subes β de 1 a 10. ¿Qué esperas?
- a) Reconstrucciones más nítidas · b) Mapa más ordenado pero imágenes más borrosas · c) Nada · d) Que el VAE deje de entrenar

**5.** Un banco entrena un autoencoder solo con transacciones normales. Una transacción nueva tiene un error de reconstrucción 40 veces mayor que el típico. ¿Qué significa?
- a) Que el modelo está roto · b) Que es probablemente una transacción anómala que conviene revisar · c) Que es fraude seguro · d) Nada

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** Por eso no necesita etiquetas (autosupervisado).
2. **b)** Sin un cuello de botella estrecho, copiar es lo más fácil.
3. **b)** El autoencoder no tiene ninguna presión para ordenar el mapa.
4. **b)** El KL pesa más: orden a costa de fidelidad.
5. **b)** Es una **alarma**, no un veredicto: lo raro no siempre es fraude (puede ser un cliente de viaje). Se revisa.
</details>

### 9.2 Preguntas para pensar (sin ejecutar)

**A.** Un compañero dice: *"Un autoencoder con `dim_latente=2` es como PCA con 2 componentes, así que da igual cuál use"*. ¿En qué tiene razón y en qué no?

**B. Caso de análisis 💼** Una fábrica toma fotos de cada tornillo que sale de la línea. Tiene 100,000 fotos de tornillos buenos y solo 30 de tornillos defectuosos (cada defecto es distinto). ¿Por qué un autoencoder puede ser mejor idea que un clasificador supervisado "bueno/defectuoso"? ¿Cómo elegirías el umbral de alarma?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

**A.** Razón: los dos comprimen a 2 números y reconstruyen; PCA **es** un autoencoder lineal. No: el autoencoder es **no lineal** y reconstruye mejor (≈ 0.046 vs. ≈ 0.054 aquí), aunque es más lento, necesita ajustar hiperparámetros y sus ejes son menos interpretables que las componentes de PCA. Si PCA basta, es la opción más simple.

**B.** Con 30 defectuosos, todos distintos, un clasificador no tiene ejemplos suficientes del "defecto" (desbalance extremo, Lección 4.2) y no reconocerá un defecto **nuevo**. Un autoencoder solo necesita aprender cómo es un tornillo **bueno**: cualquier cosa que no sepa reconstruir es sospechosa, incluso defectos nunca vistos. Umbral: mira el error en fotos buenas que no usaste para entrenar (p. ej. el percentil 99 o 99.9) y compruébalo con los 30 defectuosos; ajústalo según cuánto cuesta revisar una falsa alarma frente a dejar pasar un defecto (Lección 4.8).
</details>

### 9.3 🧩 Práctica integradora: "Restaurar documentos escaneados"

Un archivo histórico escaneó miles de formularios con dígitos escritos a mano, pero el escáner dejó **puntitos de ruido** en todas las imágenes. Te piden un modelo que las **limpie**. Usarás un **autoencoder que quita ruido** (*denoising autoencoder*): se le da la imagen **sucia** y se le pide reconstruir la **limpia**.

Lo harás en 5 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Mira el problema.** Ensuciamos los dígitos de prueba con ruido.

In [ ]:
NIVEL_RUIDO = 0.4                    # ✏️ cuánto ruido mete el escáner
torch.manual_seed(0)
sucias = (X_prueba + NIVEL_RUIDO * torch.randn_like(X_prueba)).clamp(0, 1)
mostrar(X_prueba, "Limpias (lo que queremos)")
mostrar(sucias, "Sucias (lo que tenemos)")
print(f"Error de las sucias respecto a las limpias: {((sucias - X_prueba) ** 2).mean():.4f}")

**Paso 2 🔮 — ¿Sirve un autoencoder normal?** El autoencoder de 32 números (Sección 6) se entrenó solo con imágenes **limpias**. Si le das las sucias, ¿la salida se parecerá más a la limpia o a la sucia?

✍️ *Mi apuesta:*

In [ ]:
limpiadas_normal = reconstruir(cod32, dec32, sucias)
mostrar(limpiadas_normal, "Autoencoder normal sobre imágenes sucias")
print(f"Error respecto a las limpias: {((limpiadas_normal - X_prueba) ** 2).mean():.4f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Algo ayuda (el error baja de ≈ 0.080 a ≈ 0.048), porque el cuello de botella ya descarta parte del ruido. Pero el resultado sigue manchado: nunca vio ruido al entrenar y **trata de reconstruirlo** como si fuera parte del dígito.
</details>

**Paso 3 ✏️ — Entrena un autoencoder que quita ruido.** Mismo modelo, pero ahora durante el entrenamiento la **entrada** se ensucia con `ruido=NIVEL_RUIDO` y el **objetivo** sigue siendo la imagen limpia (mira la línea `entrada = ...` de `entrenar_autoencoder`).

In [ ]:
cod_limpia, dec_limpia = entrenar_autoencoder(dim_latente=32, ruido=NIVEL_RUIDO)
limpiadas = reconstruir(cod_limpia, dec_limpia, sucias)
mostrar(sucias, "Sucias")
mostrar(limpiadas, "Autoencoder que quita ruido")
print(f"Error respecto a las limpias: {((limpiadas - X_prueba) ** 2).mean():.4f}")

**Paso 4 ✋ — Compara.** Arma una pequeña tabla con los tres errores (sucias sin tocar, autoencoder normal, autoencoder que quita ruido). ¿Cuántas veces se redujo el error respecto a las sucias?

✍️ *Mi tabla:*

<details><summary><b>👀 Ver respuesta</b></summary>

Aproximadamente:

| Versión | Error respecto a las limpias |
|---|---|
| Sucias, sin tocar | ≈ 0.080 |
| Autoencoder normal | ≈ 0.048 |
| Autoencoder que quita ruido | ≈ 0.020 |

El que quita ruido reduce el error unas **4 veces**. Entrenar con el **mismo tipo de suciedad** que habrá en la vida real es la clave.
</details>

**Paso 5 ✋ — Piensa como ingeniero/a.**
1. El escáner del archivo se cambia por uno que mete ruido **mucho más fuerte** (0.8). ¿Qué esperas del modelo actual? ✏️ Pruébalo cambiando `NIVEL_RUIDO` en el Paso 1 y ejecutando de nuevo **solo** los pasos 1 y 2, y luego la celda de abajo.
2. ¿Qué relación ves entre "aprender a quitar ruido" y los modelos de difusión que generan imágenes (Lección 13.4)?

✍️ *Mis respuestas:*

In [ ]:
# Aplica el modelo YA entrenado (con ruido 0.4) a las imágenes sucias actuales
limpiadas = reconstruir(cod_limpia, dec_limpia, sucias)
print(f"Ruido actual = {NIVEL_RUIDO} · error de las sucias = {((sucias - X_prueba) ** 2).mean():.4f} "
      f"· error tras limpiar = {((limpiadas - X_prueba) ** 2).mean():.4f}")
mostrar(limpiadas, "Limpieza con el modelo entrenado para ruido 0.4")

<details><summary><b>👀 Ver respuesta</b></summary>

1. Con ruido 0.8 el modelo entrenado con 0.4 todavía ayuda (el error baja de ≈ 0.21 a ≈ 0.06), pero queda unas **3 veces peor** que con el ruido para el que fue entrenado (≈ 0.02): las reconstrucciones salen más borrosas y se pierden trazos. Para un ruido nuevo hay que **reentrenar con ese ruido** (o con una mezcla de niveles): un modelo solo es bueno en las condiciones que vio (lo mismo que la deriva de datos, Lección 15.2).
2. ¡Es la misma idea! Un modelo de difusión es, en esencia, un **experto en quitar ruido**, entrenado con **todos** los niveles de ruido. Para generar, empieza con ruido puro y lo limpia poco a poco hasta que aparece una imagen. Lo verás en la Lección 13.4.
</details>

---
## 📚 Para profundizar
- 📺 **Video:** Arxiv Insights, "Variational Autoencoders" (inglés, con subtítulos): explicación visual del espacio latente.
- 📘 **Artículo original:** Kingma y Welling (2013), *Auto-Encoding Variational Bayes*.
- 📘 **Tutorial:** documentación de Keras/PyTorch, ejemplos "VAE on MNIST".
- 🎮 **Interactivo:** busca "latent space explorer" para jugar con el espacio latente de modelos entrenados con caras o fuentes.

---
⬅️ **Anterior:** [13.1 Discriminativo vs. generativo: el crítico y el artista](13.1-discriminativo-vs-generativo.ipynb) · ➡️ **Siguiente:** [13.3 GANs: el falsificador y el detective](13.3-gans.ipynb)